# Primer 2 of 3: NumPy (Numeric Python)

**DSC 210 Foundations of Data Science**

*A multi-day primer (~2 class days). Adapts a colleague's NumPy notebook and material from Huber (CC BY-NC-SA 4.0). Uses Palmer Penguins (CC0) plus a classic heights/weights example.*

## How to use this notebook (instructor)

Same three roles as Primer 1: **RUN-TOGETHER** (complete code, run live), **FILL-IN-LIVE** (`# together:` prompt, runs as-is), and **PREDICT / DISCUSS** (Markdown). Two days; each ends with an activity. Hide this cell in a student copy.

## Where we are

```
ASK  ->  GET  ->  [ EXPLORE ]  ->  MODEL  ->  COMMUNICATE
```

Primer 1 gave us Python. But summarizing a column meant writing `sum(x)/len(x)` by hand, and doing math to a whole list at once did not even work. **NumPy** fixes both. It is the numerical engine under nearly every data science tool in Python, including pandas, which we use next.

## What you will be able to do

1. Explain why plain lists fail for element-wise math.
2. Create NumPy **arrays** and use element-wise operations and **broadcasting**.
3. Filter data with **boolean masks**.
4. Work with **2D arrays**: shape, indexing, slicing, transpose.
5. Compute summary statistics and a **correlation**, and read the `ddof` subtlety.

---

# Day 1: Why Arrays? Vectorization and Masks

---

## 1.1 The problem with lists

In data science we constantly apply the *same* operation to a whole collection of numbers. Lists cannot do this. Here is the classic example: computing body-mass index, BMI = weight / height^2, for several people.

In [ ]:
# RUN-TOGETHER (the colleague's BMI hook: the failure IS the lesson)
height_m = [1.73, 1.68, 1.71, 1.89, 1.79]
weight_kg = [65.4, 59.2, 63.6, 88.4, 68.7]

try:
    bmi = weight_kg / height_m ** 2   # we WANT this to work
except TypeError as e:
    print('TypeError:', e)

The error is the point. For a **list**, `**` and `/` are not defined element-by-element; there is no way to say "divide each weight by each height squared" without a loop. We want something that just does the math. That is NumPy.

## 1.2 NumPy arrays to the rescue

**Definition (array).** A NumPy **array** (`ndarray`) is an ordered collection of values, like a list, but with two superpowers: (1) math happens **element-wise** across the whole array, and (2) it is fast (the looping runs in compiled C, not Python). One rule: an array is **homogeneous**, every element shares one type.

In [ ]:
# RUN-TOGETHER
import numpy as np

height_m = np.array([1.73, 1.68, 1.71, 1.89, 1.79])
weight_kg = np.array([65.4, 59.2, 63.6, 88.4, 68.7])

bmi = weight_kg / height_m ** 2   # now it works, element by element
print(bmi)
print('check first person:', 65.4 / 1.73 ** 2)

> **Predict, then run.** Lists and arrays react to `+` completely differently. Predict each result before running.

- `[1, 2, 3] + [1, 2, 3]`  -> ____
- `np.array([1,2,3]) + np.array([1,2,3])` -> ____

In [ ]:
# RUN-TOGETHER
print([1, 2, 3] + [1, 2, 3])                    # lists: glued together
print(np.array([1, 2, 3]) + np.array([1, 2, 3])) # arrays: added element-wise

## 1.3 The homogeneity rule

Because an array holds one type, mixing types forces a promotion to a common type. This is worth seeing on purpose so it never surprises you.

In [ ]:
# FILL-IN-LIVE
# together: predict the type, then run. bool + int -> ?
mixed = np.array([True, 1, 2])   # <-- discuss what type this becomes
print(mixed, '->', mixed.dtype)  # bools promoted to int

> **Discuss.** `np.array([True, 1, 2])` becomes an integer array. What do you think `np.array([1, 2, 'go'])` becomes, and why? (What single type can hold all three?)

## 1.4 Broadcasting: array meets scalar

**Broadcasting** is how NumPy applies an operation between an array and a single value: the scalar is applied to every element. Unit conversions are pure broadcasting.

In [ ]:
# RUN-TOGETHER
import seaborn as sns
penguins = sns.load_dataset('penguins')
flippers_mm = penguins['flipper_length_mm'].dropna().to_numpy()

flippers_cm = flippers_mm / 10     # every value divided by 10
print('first five (mm):', flippers_mm[:5])
print('first five (cm):', flippers_cm[:5])

## 1.5 Boolean masks: filtering without loops

Comparing an array to a value gives an array of `True`/`False`. Indexing an array *with* that boolean array keeps only the `True` positions. This replaces filtering loops and reads almost like English.

In [ ]:
# RUN-TOGETHER
# Which flippers are longer than 210 mm?
mask = flippers_mm > 210
print('first ten flags:', mask[:10])
print('how many over 210:', mask.sum())   # True counts as 1
print('their values:', flippers_mm[flippers_mm > 210][:5], '...')

> **Predict, then run.** `mask.sum()` counted the long flippers. Predict what `mask.mean()` gives, and what it *means*. (Hint: average of 0s and 1s.)

In [ ]:
# FILL-IN-LIVE
# together: fill the threshold to select flippers SHORTER than 190 mm.
short = flippers_mm[flippers_mm < 190]    # <-- fill in live
print('count under 190:', short.size)
print('fraction of all penguins:', round(short.size / flippers_mm.size, 3))

## Day 1 activity: vectorize a computation (~15 min)

> **Small-group build.** Convert a whole column at once and filter it, with no loops. Fill in the blanks; predict before running.

In [ ]:
# Group build (fill in the ____)
import seaborn as sns, numpy as np
penguins = sns.load_dataset('penguins')
mass_g = penguins['body_mass_g'].dropna().to_numpy()

# (1) Convert grams to kilograms (1000 g = 1 kg) by broadcasting.
mass_kg = mass_g / ____            # fill in
print('first five (kg):', mass_kg[:5])

# (2) How many penguins weigh more than 5 kg?
heavy = mass_kg[mass_kg ____ 5]    # fill in the comparison
print('count over 5 kg:', heavy.size)

# (3) What fraction of all penguins is that?
print('fraction:', round(heavy.size / mass_kg.size, 3))

> **Discuss to close Day 1.** Compare the array approach to how you would have done step 2 with a plain Python list. What did vectorization buy you in code length and clarity? (This is why NumPy underlies the whole ecosystem.)

---

# Day 2: 2D Arrays, Statistics, and Correlation

---

## 2.1 Two-dimensional arrays

Real data is a table: many rows, several columns. NumPy represents that as a **2D array**. Now `shape` matters and indexing takes a `[row, column]` pair. We use a classic dataset here: heights and weights of 8 baseball players, because it makes shape and transpose vivid.

In [ ]:
# RUN-TOGETHER
import numpy as np

# rows = [heights (in)], [weights (lb)]
players = np.array([[74, 74, 72, 72, 73, 69, 69, 71],       # heights
                    [180, 215, 210, 210, 188, 176, 209, 200]])  # weights

print('shape (rows, cols):', players.shape)
print('an attribute has no parentheses; a method like .mean() does')

**Definition (attribute vs. method).** `players.shape` is an **attribute**: stored information, no parentheses. `players.mean()` is a **method**: an action, with parentheses. Mixing these up is a common beginner error.

In [ ]:
# RUN-TOGETHER
# Row 0 is all heights; row 1 is all weights. Column j is one player.
print('all heights:', players[0])        # first row
print('player 0 (h, w):', players[:, 0]) # first column, both rows
print('height of player 3:', players[0, 3])  # row 0, col 3

> **Predict, then run.** `players` has shape (2, 8). After transposing with `.T` (swap rows and columns), what shape do you expect? What will one row represent then?

In [ ]:
# FILL-IN-LIVE
by_player = players.T    # transpose: now one player per row
print('transposed shape:', by_player.shape)
# together: print player 0's [height, weight] row by filling the index.
print('player 0 row:', by_player[0])   # <-- confirm this is [74, 180]

## 2.2 Summary statistics

NumPy gives `mean`, `median`, `std`, `sum`, `min`, `max`, and more. Back to penguins for a real summary.

In [ ]:
# RUN-TOGETHER
import seaborn as sns
penguins = sns.load_dataset('penguins')
mass = penguins['body_mass_g'].dropna().to_numpy()

print('count :', mass.size)
print('mean  :', round(mass.mean(), 1), 'g')
print('median:', round(np.median(mass), 1), 'g')
print('std   :', round(mass.std(), 1), 'g')

**The `ddof` subtlety (for our stats-trained students).** `np.std` divides by $n$ by default (the **population** standard deviation). The **sample** version from your stats course divides by $n-1$; request it with `ddof=1`. The gap is tiny for large $n$ but matters to know, because **pandas defaults the other way** (`ddof=1`).

In [ ]:
# RUN-TOGETHER
print('population std (ddof=0):', round(mass.std(), 2))
print('sample     std (ddof=1):', round(mass.std(ddof=1), 2))
print('n =', mass.size, '-> the two are close because n is large')

## 2.3 Correlation: do two variables move together?

`np.corrcoef` measures **linear association** between two variables, from -1 (perfect opposite) through 0 (none) to +1 (perfect together). Do heavier penguins have longer flippers?

In [ ]:
# RUN-TOGETHER
# Keep only penguins with BOTH measurements present.
clean = penguins[['body_mass_g', 'flipper_length_mm']].dropna()
mass = clean['body_mass_g'].to_numpy()
flip = clean['flipper_length_mm'].to_numpy()

R = np.corrcoef(mass, flip)    # returns a 2x2 matrix
print(R)
print('correlation r =', round(R[0, 1], 3))

> **Predict, then discuss.** The correlation comes out strongly positive (around 0.87). In plain words, what does that say about penguins? And the crucial caution: does a high correlation prove that more mass *causes* longer flippers? Give one alternative explanation. (We return to correlation-vs-causation in the EXPLORE unit.)

## Primer 2 capstone activity: standardize and find outliers (~20 min)

> **Small-group investigation.** A common data science move is the **z-score**: z = (value - mean) / std, which re-expresses each value as "how many standard deviations from the mean." Use it to flag unusual penguins. Fill in the blanks and predict before running.

In [ ]:
# Capstone build (fill in the ____)
import seaborn as sns, numpy as np
penguins = sns.load_dataset('penguins')
mass = penguins['body_mass_g'].dropna().to_numpy()

# (1) Standardize the masses to z-scores.
z = (mass - mass.mean()) / ____      # fill in the denominator

# (2) By construction, z should have mean ~0 and std ~1. Check:
print('z mean:', round(z.mean(), 10))
print('z std :', round(z.std(), 10))

# (3) Flag penguins more than 2 SD from the mean (unusually big/small).
unusual = mass[np.abs(z) ____ 2]     # fill in the comparison
print('number of unusual penguins:', unusual.size)
print('their masses:', np.sort(unusual))

> **Report out and reflect.** How many penguins did a 2-SD rule flag? Change the threshold to 1.5 and to 3, and watch the count. **Discussion:** a fixed z-score cutoff always flags *some* points as "outliers," even in perfectly normal data. What does that tell you about calling something an outlier? (This previews the outlier-detection topic later in the course.)

## Primer 2 summary

- Lists cannot do element-wise math; **arrays** can, and fast.
- Arrays are **homogeneous**; mixed inputs promote to one type.
- **Broadcasting** applies a scalar across a whole array; **boolean masks** filter without loops.
- **2D arrays** use `[row, col]` indexing; `.shape` is an attribute, `.mean()` is a method; `.T` transposes.
- `mean`, `median`, `std` summarize; watch **`ddof`** (NumPy defaults to population, pandas to sample).
- `corrcoef` measures **linear association**, not causation.

**Next: Primer 3, visualization with seaborn**, where we finally *see* the distributions and relationships we have only summarized as numbers.

## References

- Huber, F. *Hands-on Introduction to Data Science with Python*. https://florian-huber.github.io/data_science_course/
- Harris et al. (2020). *Array programming with NumPy*. Nature 585. https://doi.org/10.1038/s41586-020-2649-2
- VanderPlas, J. *Python Data Science Handbook*, ch. 2 (free): https://jakevdp.github.io/PythonDataScienceHandbook/
- NumPy broadcasting: https://numpy.org/doc/stable/user/basics.broadcasting.html